# L21 worked example: serving a model, and measuring what it costs

The model is Lecture 14's five-network ensemble for concrete compressive strength, with its
one-factor recalibration. This notebook turns it into an HTTP service and measures it:

1. **Fit and save** the ensemble, with a manifest recording versions, the data hash and the
   artifact's own hash.
2. **The service**, `app.py`: typed, range-checked requests; a response with an interval, an
   out-of-domain flag and the model version; the model loaded once; liveness and readiness.
3. **In process**, with FastAPI's test client: a good request, a 422, an out-of-domain mix.
4. **On a port**, with uvicorn in the background: latency and throughput against concurrency,
   checked with Little's law.
5. **Batching**: one mix per request against a thousand.
6. **A deliberate mistake**: the same prediction inside `async def`, and what it does to a health
   check.
7. **An optimizer as the client**, counting its calls.
8. **Cost** per prediction and per LLM answer.
9. **Docker**, only if it is installed: the image, its size, and a secret recovered from a layer.

Everything runs on this machine. The client and the server share it, so every latency here is a
floor: a real network adds to each one.

## Run this first on Colab

Colab starts from its own preinstalled environment rather than this course's `uv`
environment, so run the cell below before anything else. It installs what this
notebook needs and Colab does not already have. Outside Colab it does nothing, so
you can run it or skip it.


In [ ]:
# Run this first on Colab. Anywhere else this cell does nothing.
#
# Only genuinely missing packages are installed, so Colab's own versions of
# everything it already ships are left alone.
import importlib.util
import subprocess
import sys

REQUIREMENTS = {
    "app": "app",
    "fastapi": "fastapi",
    "httpx": "httpx",
    "joblib": "joblib",
    "numpy": "numpy",
    "pandas": "pandas",
    "pydantic": "pydantic",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "uvicorn": "uvicorn",
    "xlrd": "xlrd",
}


def _missing(module):
    try:
        return importlib.util.find_spec(module) is None
    except ModuleNotFoundError:  # the parent package is absent
        return True


if "google.colab" in sys.modules:
    need = sorted({pip for mod, pip in REQUIREMENTS.items() if _missing(mod)})
    if need:
        print("installing:", " ".join(need))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Colab setup done." if need else "Colab: nothing to install.")


In [2]:
import hashlib, io, json, os, platform, shutil, socket, subprocess, sys, tempfile, time
import urllib.request, warnings, zipfile
from pathlib import Path

import httpx
import joblib
import numpy as np
import pandas as pd
import sklearn
import fastapi
import uvicorn  # started below as a separate process; imported so the version is on record

WORK = Path("l21_work")
(WORK / "model").mkdir(parents=True, exist_ok=True)
print("python", platform.python_version(), "| scikit-learn", sklearn.__version__,
      "| fastapi", fastapi.__version__, "| uvicorn", uvicorn.__version__)
print("machine", platform.machine(), platform.system(), "| cores", os.cpu_count())

python 3.12.11 | scikit-learn 1.7.2 | fastapi 0.121.1 | uvicorn 0.38.0
machine arm64 Darwin | cores 14


## 1. Fit and save the model

Lecture 14's grouped split (mixes, not rows, are kept apart), its network (Lecture 9's one hidden
layer of 16 tanh units), seeds 0 to 4, and one scale factor fitted on calibration mixes. The
manifest is what lets the service prove that the file it loads is the file that was tested.

The networks stop at their 5,000-iteration cap without converging, and the warning is silenced
as in Lecture 14. That is why your scale factor may differ from the notes' 2.09 and from Lecture
14's 1.66: an unconverged fit depends on floating-point details that library builds change. The
saved artifact is the model; rerunning the recipe gives a slightly different one.

In [3]:
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore", category=ConvergenceWarning)

UCI = "https://archive.ics.uci.edu/static/public/165/concrete+compressive+strength.zip"
COLUMNS = ["cement", "slag", "fly_ash", "water", "superplasticizer",
           "coarse_agg", "fine_agg", "age_days", "strength_mpa"]
FEATURES, MIX = COLUMNS[:8], COLUMNS[:7]
xls = WORK / "Concrete_Data.xls"
if not xls.exists():
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(UCI).read()))
    xls.write_bytes(z.read("Concrete_Data.xls"))
df = pd.read_excel(xls)
df.columns = COLUMNS
X, y = df[FEATURES].to_numpy(), df["strength_mpa"].to_numpy()
groups = df.groupby(MIX).ngroup().to_numpy()

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, y, groups))
fit_i, cal_i = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
                    .split(X[tr], y[tr], groups[tr]))
fit_i, cal_i = tr[fit_i], tr[cal_i]
nets = [make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(16,), activation="tanh",
        solver="lbfgs", max_iter=5000, random_state=s)).fit(X[fit_i], y[fit_i]) for s in range(5)]

def ensemble(Xq):
    P = np.array([net.predict(Xq) for net in nets])
    return P.mean(0), P.std(0, ddof=1)

mc, sc = ensemble(X[cal_i])
scale = float(np.sqrt(np.mean(((y[cal_i] - mc) / sc) ** 2)))
mt, st = ensemble(X[te])
Z90 = 1.6448536269514722
print(f"fit {len(fit_i)}, calibration {len(cal_i)}, test {len(te)} rows")
print(f"iterations used: {[n[-1].n_iter_ for n in nets]} (cap 5000)")
print(f"scale factor s = {scale:.2f}; test RMSE {np.sqrt(np.mean((y[te] - mt) ** 2)):.2f} MPa; "
      f"90% interval coverage on test mixes {np.mean(np.abs(y[te] - mt) <= Z90 * scale * st):.1%}")

artifact = WORK / "model" / "ensemble.joblib"
joblib.dump(nets, artifact)
manifest = dict(model_version="concrete-ens5-v1", sklearn=sklearn.__version__, numpy=np.__version__,
                python=platform.python_version(),
                data_sha256=hashlib.sha256(df.to_csv(index=False).encode()).hexdigest(),
                split="Lecture 14 grouped split, seeds 42 and 0", scale=scale,
                ranges={f: [float(X[tr][:, i].min()), float(X[tr][:, i].max())]
                        for i, f in enumerate(FEATURES)},
                sha256=hashlib.sha256(artifact.read_bytes()).hexdigest())
(WORK / "model" / "manifest.json").write_text(json.dumps(manifest, indent=1))
print(f"wrote {artifact} ({artifact.stat().st_size:,} bytes) and manifest.json")
test_mixes = [dict(zip(FEATURES, map(float, row))) for row in X[te]]

fit 625, calibration 210, test 195 rows
iterations used: [5000, 5000, 5000, 5000, 5000] (cap 5000)
scale factor s = 2.09; test RMSE 5.29 MPa; 90% interval coverage on test mixes 95.9%
wrote l21_work/model/ensemble.joblib (45,825 bytes) and manifest.json


## 2. The service

This is `lectures/l21/service/app.py` from the course repository, written to disk unchanged. Read
the `lifespan` function (the model is loaded once, and refused if its hash or scikit-learn
version does not match the manifest), the `Mix` bounds (impossible inputs are rejected), and the
`Prediction` fields (the interval, the domain flag and the version travel with every answer).

In [4]:
%%writefile l21_work/app.py
"""Lecture 14's concrete strength ensemble, served over HTTP. The L21 reference service.

    uv run --project . uvicorn app:app --port 8000        # from this directory
    curl -s localhost:8000/readyz

The model directory (MODEL_DIR, default ./model) holds two files written by
lectures/l21/figures/make_figures.py:

    ensemble.joblib   five scikit-learn pipelines (Lecture 9's network, seeds 0 to 4)
    manifest.json     versions, the training ranges, the recalibration factor, and the
                      SHA-256 of ensemble.joblib

Two endpoints are deliberately wrong and exist only so the lecture can measure what they do:
/predict_batch_async runs a CPU-bound prediction inside `async def`. Do not copy it.
"""
from __future__ import annotations

import hashlib
import json
import os
import time
from contextlib import asynccontextmanager
from pathlib import Path

import joblib
import numpy as np
import sklearn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

MODEL_DIR = Path(os.environ.get("MODEL_DIR", "model"))
Z90 = 1.6448536269514722          # two-sided 90% normal quantile
FEATURES = ["cement", "slag", "fly_ash", "water", "superplasticizer",
            "coarse_agg", "fine_agg", "age_days"]
STATE: dict = {}


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Load the model once, before the server accepts a single request.

    Fail loudly rather than serve a model that is not the one that was tested: the artifact's
    hash and the scikit-learn version are both checked against the manifest.
    """
    t0 = time.perf_counter()
    manifest = json.loads((MODEL_DIR / "manifest.json").read_text())
    artifact = MODEL_DIR / "ensemble.joblib"
    if sha256(artifact) != manifest["sha256"]:
        raise RuntimeError("ensemble.joblib does not match the hash in manifest.json")
    if sklearn.__version__ != manifest["sklearn"]:
        raise RuntimeError(f"model saved with scikit-learn {manifest['sklearn']}, "
                           f"this process has {sklearn.__version__}")
    STATE["nets"] = joblib.load(artifact)   # a pickle: load only artifacts you produced
    STATE["manifest"] = manifest
    STATE["lo"] = np.array([manifest["ranges"][f][0] for f in FEATURES])
    STATE["hi"] = np.array([manifest["ranges"][f][1] for f in FEATURES])
    STATE["load_s"] = time.perf_counter() - t0
    yield
    STATE.clear()


app = FastAPI(title="Concrete strength surrogate", version="1.0.0", lifespan=lifespan)


class Mix(BaseModel):
    """One concrete mix. kg per cubic metre of mixture, and age in days.

    The bounds reject inputs that are physically impossible (a negative mass, an age of zero).
    They do not reject inputs that are merely unusual: that is the in_domain flag's job.
    """
    cement: float = Field(gt=0, le=1000)
    slag: float = Field(ge=0, le=1000)
    fly_ash: float = Field(ge=0, le=1000)
    water: float = Field(gt=0, le=1000)
    superplasticizer: float = Field(ge=0, le=100)
    coarse_agg: float = Field(ge=0, le=2000)
    fine_agg: float = Field(ge=0, le=2000)
    age_days: float = Field(ge=1, le=3650)


class Prediction(BaseModel):
    strength_mpa: float
    interval_90: tuple[float, float]
    in_domain: bool
    outside: list[str]
    model_version: str


def predict_rows(X: np.ndarray) -> list[Prediction]:
    """Ensemble mean, and a 90% interval from the recalibrated ensemble spread (Lecture 14)."""
    P = np.array([net.predict(X) for net in STATE["nets"]])
    mu, sd = P.mean(0), STATE["manifest"]["scale"] * P.std(0, ddof=1)
    out = (X < STATE["lo"]) | (X > STATE["hi"])
    version = STATE["manifest"]["model_version"]
    return [Prediction(strength_mpa=round(float(m), 2),
                       interval_90=(round(float(m - Z90 * s), 2), round(float(m + Z90 * s), 2)),
                       in_domain=not o.any(),
                       outside=[f for f, flag in zip(FEATURES, o) if flag],
                       model_version=version)
            for m, s, o in zip(mu, sd, out)]


def as_matrix(mixes: list[Mix]) -> np.ndarray:
    return np.array([[getattr(m, f) for f in FEATURES] for m in mixes])


@app.get("/healthz")
def healthz() -> dict:
    """Liveness: the process is running and answering. Says nothing about the model."""
    return {"status": "alive"}


@app.get("/readyz")
def readyz() -> dict:
    """Readiness: the model is loaded and this process can serve predictions."""
    if "nets" not in STATE:
        raise HTTPException(status_code=503, detail="model not loaded")
    m = STATE["manifest"]
    return {"status": "ready", "model_version": m["model_version"], "sklearn": m["sklearn"],
            "load_seconds": round(STATE["load_s"], 4)}


@app.post("/predict")
def predict(mix: Mix) -> Prediction:
    return predict_rows(as_matrix([mix]))[0]


@app.post("/predict_batch")
def predict_batch(mixes: list[Mix]) -> list[Prediction]:
    if len(mixes) > 10_000:
        raise HTTPException(status_code=413, detail="at most 10000 mixes per request")
    return predict_rows(as_matrix(mixes))


@app.post("/predict_batch_async")
async def predict_batch_async(mixes: list[Mix]) -> list[Prediction]:
    """DELIBERATELY WRONG. The same CPU-bound work as /predict_batch, inside `async def`, so it
    runs on the event loop and every other request, /healthz included, waits until it ends."""
    return predict_rows(as_matrix(mixes))

Overwriting l21_work/app.py


## 3. In process, with the test client

`TestClient` runs the app inside this Python process, including its lifespan, with no server and
no port. It is how you test the contract before anything is deployed.

In [5]:
from fastapi.testclient import TestClient

os.environ["MODEL_DIR"] = str(WORK / "model")
sys.path.insert(0, str(WORK))
import app as service

with TestClient(service.app) as c:
    print("readyz      ", c.get("/readyz").json())
    good = test_mixes[0]
    print("predict     ", c.post("/predict", json=good).json())
    r = c.post("/predict", json={**good, "cement": -50})
    print("cement = -50", r.status_code, r.json()["detail"][0]["msg"])
    odd = {**good, "slag": 400.0, "age_days": 730.0}
    print("outside     ", c.post("/predict", json=odd).json())
    print("paths in the OpenAPI schema:", sorted(c.get("/openapi.json").json()["paths"]))

readyz       {'status': 'ready', 'model_version': 'concrete-ens5-v1', 'sklearn': '1.7.2', 'load_seconds': 0.0018}
predict      {'strength_mpa': 48.34, 'interval_90': [30.04, 66.64], 'in_domain': True, 'outside': [], 'model_version': 'concrete-ens5-v1'}
cement = -50 422 Input should be greater than 0
outside      {'strength_mpa': 58.92, 'interval_90': [41.82, 76.01], 'in_domain': False, 'outside': ['slag', 'age_days'], 'model_version': 'concrete-ens5-v1'}
paths in the OpenAPI schema: ['/healthz', '/predict', '/predict_batch', '/predict_batch_async', '/readyz']


The impossible mix gets a 422 that names the field and the rule. The unusual one gets an answer,
flagged: look at its interval. It is no wider than the in-domain mix's, which is Lecture 14's point
about ensembles off their data, and the reason the flag exists.

## 4. On a port, under load

Now the real thing: uvicorn in a separate process, listening on `127.0.0.1`. The notebook waits
for `/readyz`, which answers only once the model is loaded.

In [6]:
def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def start_server(workers=1):
    port = free_port()
    t0 = time.perf_counter()
    proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1",
                             "--port", str(port), "--workers", str(workers), "--log-level", "warning"],
                            cwd=WORK, env={**os.environ, "MODEL_DIR": "model"})
    while True:
        try:
            r = httpx.get(f"http://127.0.0.1:{port}/readyz", timeout=1)
            if r.status_code == 200:
                return proc, port, time.perf_counter() - t0, r.json()
        except httpx.HTTPError:
            pass
        if proc.poll() is not None:
            raise RuntimeError("uvicorn exited before it was ready")
        time.sleep(0.02)

server, PORT, t_ready, ready = start_server()
BASE = f"http://127.0.0.1:{PORT}"
print(f"ready after {t_ready:.2f} s, of which loading the model took {ready['load_seconds']:.3f} s")

ready after 0.84 s, of which loading the model took 0.056 s


The load test uses **closed-loop** clients: each sends a request, waits for the answer, and sends
the next. Each client is its own process. A first version that ran every client from one Python
process measured the client's limit rather than the server's, so the clients are a small script
started as separate processes.

In [7]:
(WORK / "load_client.py").write_text('''
import json, sys, time, httpx
port, path, n, start_at, bodies = sys.argv[1], sys.argv[2], int(sys.argv[3]), float(sys.argv[4]), json.load(open(sys.argv[5]))
lat = []
with httpx.Client(base_url=f"http://127.0.0.1:{port}", timeout=30) as c:
    c.post(path, json=bodies[0])
    while time.time() < start_at:
        time.sleep(0.001)
    t0 = time.time()
    for i in range(n):
        t = time.perf_counter()
        c.post(path, json=bodies[i % len(bodies)]).raise_for_status()
        lat.append(time.perf_counter() - t)
print(json.dumps({"lat": lat, "t0": t0, "t1": time.time()}))
''')
(WORK / "mixes.json").write_text(json.dumps(test_mixes))

def load(port, concurrency, n_each, path="/predict"):
    start_at = time.time() + 1.0 + 0.05 * concurrency
    procs = [subprocess.Popen([sys.executable, str(WORK / "load_client.py"), str(port), path,
                               str(n_each), str(start_at), str(WORK / "mixes.json")],
                              stdout=subprocess.PIPE, text=True) for _ in range(concurrency)]
    parts = [json.loads(p.communicate()[0]) for p in procs]
    lat = np.concatenate([p["lat"] for p in parts]) * 1e3
    wall = max(p["t1"] for p in parts) - min(p["t0"] for p in parts)
    rate = len(lat) / wall
    return dict(clients=concurrency, p50_ms=np.percentile(lat, 50), p95_ms=np.percentile(lat, 95),
                mean_ms=lat.mean(), throughput_rps=rate, littles_L=rate * lat.mean() / 1e3)

rows = [load(PORT, c, max(100, 800 // c)) for c in (1, 2, 4, 8, 16)]
print(pd.DataFrame(rows).round(2).to_string(index=False))

 clients  p50_ms  p95_ms  mean_ms  throughput_rps  littles_L
       1    0.87    1.27     0.92         1082.66       1.00
       2    1.32    1.69     1.39         1435.27       2.00
       4    2.33    2.85     2.37         1681.34       3.98
       8    4.38    5.51     4.45         1783.89       7.94
      16    8.93   11.30     9.15         1720.77      15.75


Two things to read in that table. First, **Little's law**, $L = \lambda W$: the last column,
throughput times mean latency, should come back as the number of clients, because each client
always has exactly one request in the system. Second, **saturation**: once throughput stops
rising, every extra client only waits, and the mean latency grows as clients divided by
throughput. Where that happens depends on your machine.

## 5. One request at a time, against a batch

In [8]:
with httpx.Client(base_url=BASE) as c:
    for B in (1, 10, 100, 1000):
        body = (test_mixes * (B // len(test_mixes) + 1))[:B]
        reps = 100 if B <= 100 else 20
        c.post("/predict_batch", json=body)
        t = time.perf_counter()
        for _ in range(reps):
            c.post("/predict_batch", json=body).raise_for_status()
        per = (time.perf_counter() - t) / reps
        print(f"{B:5d} mixes per request: {per * 1e3:7.2f} ms per request, "
              f"{per / B * 1e6:7.1f} us per prediction")

    1 mixes per request:    0.93 ms per request,   934.5 us per prediction


   10 mixes per request:    1.01 ms per request,   101.5 us per prediction


  100 mixes per request:    1.60 ms per request,    16.0 us per prediction


 1000 mixes per request:    8.31 ms per request,     8.3 us per prediction


## 6. A deliberate mistake: CPU work inside `async def`

**This cell is meant to show a failure.** `/predict_batch` is a plain `def`, so FastAPI runs it in
a thread pool. `/predict_batch_async` does exactly the same work inside `async def`, so it runs on
the event loop and nothing else can be answered until it finishes. Four threads send 5,000-mix
batches to one endpoint while a probe asks `/healthz` every 50 ms.

In [9]:
import threading

def stall(path, seconds=3.0):
    heavy = (test_mixes * 40)[:5000]
    stop = time.perf_counter() + seconds
    probe = []
    def hammer():
        with httpx.Client(base_url=BASE, timeout=60) as c:
            while time.perf_counter() < stop:
                c.post(path, json=heavy).raise_for_status()
    threads = [threading.Thread(target=hammer) for _ in range(4)]
    for t in threads:
        t.start()
    with httpx.Client(base_url=BASE, timeout=60) as c:
        while time.perf_counter() < stop:
            t0 = time.perf_counter()
            c.get("/healthz").raise_for_status()
            probe.append((time.perf_counter() - t0) * 1e3)
            time.sleep(0.05)
    for t in threads:
        t.join()
    return len(probe), np.percentile(probe, 50), np.percentile(probe, 95)

for path in ("/predict_batch", "/predict_batch_async"):
    k, p50, p95 = stall(path)
    print(f"{path:22s} health checks answered: {k:3d}   p50 {p50:6.1f} ms   p95 {p95:6.1f} ms")

/predict_batch         health checks answered:  30   p50   52.0 ms   p95   81.8 ms


/predict_batch_async   health checks answered:  14   p50  160.9 ms   p95  175.8 ms


## 7. An optimizer as the client

A design loop is the surrogate's real caller. Here Nelder-Mead searches cement and water for the
strongest mix at 28 days, holding the other inputs at a typical mix, and treats an out-of-domain
answer as unusable. Every evaluation is one HTTP request. Then the same question is answered with
2,000 random candidates sent in two batches. Count the calls.

In [10]:
from scipy.optimize import minimize

base_mix = dict(cement=300.0, slag=100.0, fly_ash=0.0, water=180.0, superplasticizer=6.0,
                coarse_agg=970.0, fine_agg=775.0, age_days=28.0)
calls = 0
with httpx.Client(base_url=BASE) as c:
    def objective(v):
        global calls
        calls += 1
        mix = {**base_mix, "cement": float(np.clip(v[0], 1, 1000)), "water": float(np.clip(v[1], 1, 1000))}
        p = c.post("/predict", json=mix).json()
        return -p["strength_mpa"] if p["in_domain"] else 1e3
    t = time.perf_counter()
    res = minimize(objective, x0=[300.0, 180.0], method="Nelder-Mead",
                   options=dict(xatol=1.0, fatol=0.05, maxfev=400))
    t_nm = time.perf_counter() - t
    print(f"Nelder-Mead: {calls} requests, {t_nm * 1e3:.0f} ms, best {-res.fun:.1f} MPa "
          f"at cement {res.x[0]:.0f}, water {res.x[1]:.0f} kg/m3")

    rng = np.random.default_rng(0)
    lo, hi = manifest["ranges"]["cement"], manifest["ranges"]["water"]
    cand = [{**base_mix, "cement": float(a), "water": float(b)}
            for a, b in zip(rng.uniform(*lo, 2000), rng.uniform(*hi, 2000))]
    t = time.perf_counter()
    preds = c.post("/predict_batch", json=cand[:1000]).json() + c.post("/predict_batch", json=cand[1000:]).json()
    t_b = time.perf_counter() - t
    best = max((p for p in preds if p["in_domain"]), key=lambda p: p["strength_mpa"])
    i = preds.index(best)
    print(f"Random search, batched: 2 requests for 2,000 candidates, {t_b * 1e3:.0f} ms, best "
          f"{best['strength_mpa']:.1f} MPa at cement {cand[i]['cement']:.0f}, water {cand[i]['water']:.0f} kg/m3")

Nelder-Mead: 55 requests, 52 ms, best 83.3 MPa at cement 540, water 148 kg/m3
Random search, batched: 2 requests for 2,000 candidates, 17 ms, best 83.2 MPa at cement 538, water 147 kg/m3


The optimizer's answer and the batch's answer should be close. The difference is the number of
round trips, and therefore how much of the time went to overhead rather than to the model.

## 8. Cost

Two estimates. For the CPU service, throughput is the denominator: predictions per core-hour from
what you measured above. For a hosted LLM, input and output tokens are priced separately. The
prices below were read from Anthropic's pricing page on 2026-11-04; read the page again before you
rely on them, and record the date with your estimate.

In [11]:
peak = max(r["throughput_rps"] for r in rows[:1])  # one client, one worker
print(f"one worker, one mix per request, one client: {peak:,.0f} requests/s "
      f"= {peak * 3600 / 1e6:.1f} million predictions per hour")

PRICES_2026_11_04 = {"Claude Sonnet 5.5": (2.00, 10.00)}  # USD per million tokens, (input, output)

def llm_cost(tokens_in, tokens_out, model="Claude Sonnet 5.5"):
    p_in, p_out = PRICES_2026_11_04[model]
    return tokens_in * p_in / 1e6 + tokens_out * p_out / 1e6

per = llm_cost(3000, 300)
print(f"an answer with 3,000 tokens of retrieved context and a 300-token reply: ${per:.4f}, "
      f"${per * 1000:.2f} per thousand answers")

one worker, one mix per request, one client: 1,083 requests/s = 3.9 million predictions per hour
an answer with 3,000 tokens of retrieved context and a 300-token reply: $0.0090, $9.00 per thousand answers


## 9. Docker, if it is installed

These cells build the repository's own `service/Dockerfile`, so they need a checkout of the course
repository (the notebook beside `service/`) and Docker. In Colab, or without Docker, they say so
and do nothing. Without BuildKit, the build swaps the Dockerfile's cache and bind mounts for plain
`COPY`, which produces the same final image.

In [12]:
SERVICE = Path("service")
have_docker = shutil.which("docker") is not None and SERVICE.joinpath("Dockerfile").exists()
if not have_docker:
    print("Docker or the repository's service/ directory is not available here; skipping.")
else:
    ctx = Path(tempfile.mkdtemp()) / "ctx"
    shutil.copytree(SERVICE, ctx, ignore=shutil.ignore_patterns(".venv", "__pycache__", "model"))
    shutil.copytree(WORK / "model", ctx / "model")
    if subprocess.run(["docker", "buildx", "version"], capture_output=True).returncode != 0:
        df_ = (ctx / "Dockerfile").read_text()
        i, j = df_.index("RUN --mount=type=cache"), df_.index("--no-install-project") + len("--no-install-project")
        (ctx / "Dockerfile").write_text(df_[:i] + "COPY uv.lock pyproject.toml ./\nRUN uv sync --locked --no-dev --no-install-project" + df_[j:])
    t = time.perf_counter()
    subprocess.run(["docker", "build", "-q", "-t", "l21-concrete:nb", "."], cwd=ctx, check=True, capture_output=True)
    size = int(subprocess.run(["docker", "image", "inspect", "l21-concrete:nb", "--format", "{{.Size}}"],
                              capture_output=True, text=True).stdout)
    # With Docker's containerd image store this Size is the compressed size, the bytes a machine
    # downloads; the layer sizes printed by docker history below are unpacked.
    print(f"built in {time.perf_counter() - t:.1f} s; image {size / 1e6:.0f} MB compressed")
    print(subprocess.run(["docker", "history", "l21-concrete:nb", "--format", "{{.Size}}\t{{.CreatedBy}}"],
                         capture_output=True, text=True).stdout[:1500])

built in 11.0 s; image 126 MB compressed
0B	/bin/sh -c #(nop)  CMD ["uvicorn" "app:app" …
0B	/bin/sh -c #(nop)  HEALTHCHECK &{Test:[CMD p…
0B	/bin/sh -c #(nop)  EXPOSE 8000
0B	/bin/sh -c #(nop)  USER app
0B	/bin/sh -c #(nop)  ENV PATH=/app/.venv/bin:/…
65.5kB	/bin/sh -c #(nop) COPY dir:a319cc6d0a62b095d…
16.4kB	/bin/sh -c #(nop) COPY file:e25b7e51ffb63309…
301MB	/bin/sh -c #(nop) COPY dir:3184f1e3b1017fe7e…
8.19kB	/bin/sh -c #(nop) WORKDIR /app
69.6kB	/bin/sh -c useradd --create-home --uid 10001…
0B	CMD ["python3"]
16.4kB	RUN /bin/sh -c set -eux;  for src in idle3 p…
44.7MB	RUN /bin/sh -c set -eux;   savedAptMark="$(a…
0B	ENV PYTHON_SHA256=b5796915dc45ff745cfb4e990a…
0B	ENV PYTHON_VERSION=3.12.16
0B	ENV GPG_KEY=7169605F62C751356D054A26A821E680…
4.97MB	RUN /bin/sh -c set -eux;  apt-get update;  a…
0B	ENV LANG=C.UTF-8
0B	ENV PATH=/usr/local/bin:/usr/local/sbin:/usr…
110MB	# debian.sh --arch 'arm64' out/ 'trixie' '@1…



**This cell is meant to show a failure.** A fake key passed as a build argument appears in
`docker history`; a key file copied in and deleted in the next step is gone from the running
container and still present in the saved image's earlier layer.

In [13]:
import tarfile
if not have_docker:
    print("skipping")
else:
    d = Path(tempfile.mkdtemp())
    fake = "demo-not-a-real-key-7f3a"
    (d / "key.txt").write_text(fake + "\n")
    (d / "Dockerfile").write_text("FROM python:3.12-slim\nARG API_KEY\nRUN echo configured > /tmp/marker\n"
                                  "COPY key.txt /app/key.txt\nRUN rm /app/key.txt\n")
    subprocess.run(["docker", "build", "-q", "--build-arg", f"API_KEY={fake}", "-t", "l21-leak", "."],
                   cwd=d, check=True, capture_output=True)
    hist = subprocess.run(["docker", "history", "--no-trunc", "--format", "{{.CreatedBy}}", "l21-leak"],
                          capture_output=True, text=True).stdout
    print("build argument in docker history:", fake in hist)
    print("key.txt in the running container:", "key.txt" in subprocess.run(
        ["docker", "run", "--rm", "l21-leak", "ls", "/app"], capture_output=True, text=True).stdout)
    subprocess.run(["docker", "save", "-o", str(d / "img.tar"), "l21-leak"], check=True)
    found = False
    with tarfile.open(d / "img.tar") as outer:
        for m in outer.getmembers():
            if m.isfile():
                try:
                    with tarfile.open(fileobj=io.BytesIO(outer.extractfile(m).read())) as layer:
                        for lm in layer.getmembers():
                            if lm.name.endswith("app/key.txt") and lm.isfile():
                                found = fake in layer.extractfile(lm).read().decode()
                except tarfile.ReadError:
                    pass
    print("key.txt recoverable from a saved layer:", found)
    subprocess.run(["docker", "rmi", "-f", "l21-leak", "l21-concrete:nb"], capture_output=True)

build argument in docker history: True
key.txt in the running container: False


key.txt recoverable from a saved layer: True


In [14]:
server.terminate()
server.wait(timeout=10)
print("server stopped")

server stopped


## Try it

- Start the server with `start_server(workers=4)` and rerun the load table. Where does throughput
  stop rising now, and what does `ps` say about memory?
- Change `/predict` from `def` to `async def` in `l21_work/app.py`, restart, and rerun section 4.
  What happens to p95 at 8 clients?
- Edit one value in `l21_work/model/manifest.json` and restart the server. What happens, and why
  is that the behaviour you want?
- Send the optimizer of section 7 a mix with `age_days` 730. What does the service say, and what
  should the optimizer do with it?
- Raise the 2,000 candidates to 20,000 in one request. Which limit stops you, and why does the
  service have it?